# 00 — Stage offline assets (ONLINE, CPU, internet ON)

Builds the datasets the **offline** Blackwell notebooks depend on. Run once; re-run only
when a dependency version changes.

| attach as input | produces (create dataset from output) |
|---|---|
| `behaviorsense-code` (the repo, uploaded via Kaggle CLI — see docs/07_kaggle_plan.md) | `behaviorsense-wheels`, `behaviorsense-weights` |

Why this notebook exists: the training GPU (RTX PRO 6000 Blackwell, sm_120) runs with
**no internet**, and needs torch >= 2.7 cu128. If the offline image lacks it, there is no
way to fetch it from inside the session — so every wheel and weight is staged here, and
notebook 03 decides at runtime whether to use the staged torch or the preinstalled one.

In [ ]:
# torch cu128 — the build carrying sm_120 (Blackwell) kernels. Staged even though the
# offline image may already have it: notebook 03 checks torch.cuda.get_arch_list() first
# and only installs from here if sm_120 is missing.
#
# Everything below is about making pip's resolver deterministic, because it has failed
# here twice in ways that each cost a session:
#   1. Unpinned, pip backtracks across every cu128 torch (2.7 → 2.11) hunting for a
#      satisfiable nvidia-* set, then reports ResolutionImpossible. So: pin it.
#   2. Pinning alone is NOT enough. torch 2.7.1+cu128 requires nvidia-cudnn-cu12==9.7.1.26,
#      which publishes ONLY a manylinux_2_27_x86_64 wheel. A --platform list of
#      {manylinux2014, manylinux_2_28} excludes that tag, the candidate set goes empty,
#      and pip backtracks into the same error with a different cause. Guessing individual
#      tags does not scale — the cu128 index alone publishes manylinux_2_5 / 2_12 / 2_18 /
#      2_25 / 2_27 / 2_28 across torch's dependency set — so PLATFORMS enumerates the whole
#      glibc ladder, OLDEST FIRST. pip prefers earlier entries, so when a package ships
#      several wheels it stages the most portable one, which is the right default for an
#      offline image whose glibc is not knowable from here.
#
# Wheels for BOTH 3.11 and 3.12: the offline image's interpreter is not knowable from
# here either, and missing it by one minor version costs a full 12-hour session. The
# nvidia-* dependencies are py3-none wheels, so they are shared, not duplicated.
import subprocess, pathlib
WHEELS = pathlib.Path("/kaggle/working/wheels"); WHEELS.mkdir(parents=True, exist_ok=True)

TORCH_PIN = "torch==2.7.1"          # satisfies the >=2.7 floor in requirements-train.txt
PLATFORMS = (["manylinux1_x86_64", "manylinux2010_x86_64", "manylinux2014_x86_64"]
             + [f"manylinux_2_{m}_x86_64" for m in range(5, 40)]
             + ["linux_x86_64"])
PLAT_ARGS = [a for p in PLATFORMS for a in ("--platform", p)]

for pyver in ("3.11", "3.12"):
    subprocess.run(
        ["pip", "download", TORCH_PIN, "-d", str(WHEELS),
         "--index-url", "https://download.pytorch.org/whl/cu128",
         "--python-version", pyver, "--only-binary=:all:", *PLAT_ARGS],
        check=True)
print(len(list(WHEELS.glob("*.whl"))), "wheels so far")

In [ ]:
# CPU-side packages. `accelerate` (and some outlines builds) declare a torch dependency,
# so an unconstrained download here quietly fetches the *PyPI* torch as well — verified:
# an unconstrained resolve of this list picks torch 2.13.0, another ~1 GB per python
# version, built without sm_120 and therefore useless on Blackwell, plus a second set of
# nvidia-* pins for the resolver to fight with.
#
# The constraints file pins torch to the version staged in the previous cell, and
# --find-links lets pip satisfy that from the local cu128 wheel instead of the network.
# (PEP 440: a specifier carrying no local label ignores local labels when matching, so
# `torch==2.7.1` does match `2.7.1+cu128`, and the local build sorts higher than PyPI's.
# Confirmed by dry-run: with the constraint the resolve picks 2.7.1+cu128, without it 2.13.0.)
#
# The floors below are the ones the code actually needs, not decoration: reporter.py calls
# outlines.from_transformers and outlines.types.json_schema, which exist only in outlines
# 1.x, so a 0.x resolve would import-error inside the offline session.
import subprocess, pathlib
WHEELS = pathlib.Path("/kaggle/working/wheels")
TORCH_PIN = "torch==2.7.1"
PLATFORMS = (["manylinux1_x86_64", "manylinux2010_x86_64", "manylinux2014_x86_64"]
             + [f"manylinux_2_{m}_x86_64" for m in range(5, 40)]
             + ["linux_x86_64"])
PLAT_ARGS = [a for p in PLATFORMS for a in ("--platform", p)]

CONSTRAINTS = pathlib.Path("/tmp/constraints.txt")
CONSTRAINTS.write_text("\n".join([TORCH_PIN, "outlines>=1.0", "transformers>=4.44"]) + "\n")
CPU_PKGS = ["numpy", "pydantic", "PyYAML", "Pillow",
            "transformers", "accelerate", "outlines", "safetensors"]
for pyver in ("3.11", "3.12"):
    subprocess.run(
        ["pip", "download", *CPU_PKGS, "-d", str(WHEELS),
         "--constraint", str(CONSTRAINTS), "--find-links", str(WHEELS),
         "--python-version", pyver, "--only-binary=:all:", *PLAT_ARGS],
        check=True)
total = sum(f.stat().st_size for f in WHEELS.glob("*"))
print(f"{len(list(WHEELS.glob('*.whl')))} wheels, {total/1e9:.1f} GB")

In [ ]:
# Pose-extraction packages, so shard extraction can also run OFFLINE on the Blackwell.
#
# Why this cell exists: extraction was originally online-only, purely because it downloads
# Charades. That forced ~20-30 h of RTMO inference onto a T4 while the fast card sat idle,
# and re-downloaded 13 GB at the start of each of the 3-4 sessions with the GPU doing
# nothing. Notebook 01a stages the archive once; these wheels are what let notebook 01
# then run with the internet off.
#
# onnxruntime-gpu, not onnxruntime: the CPU build silently ignores device="cuda" and runs
# RTMO on four vCPUs, which turns a 12 h session into something like a 200 h one. The
# offline notebook asserts CUDAExecutionProvider is actually present rather than trusting
# the install.
#
# The CUDA 12 build links cuDNN 9, which is already staged as a torch dependency, so no
# extra nvidia-* pins are needed here.
EXTRACT_PKGS = ["rtmlib", "onnxruntime-gpu==1.26.0", "opencv-python-headless"]
for pyver in ("3.11", "3.12"):
    subprocess.run(
        ["pip", "download", *EXTRACT_PKGS, "-d", str(WHEELS),
         "--constraint", str(CONSTRAINTS), "--find-links", str(WHEELS),
         "--python-version", pyver, "--only-binary=:all:", *PLAT_ARGS],
        check=True)
have = {f.name.split("-")[0].lower().replace("_", "-") for f in WHEELS.glob("*.whl")}
for pkg in ("onnxruntime-gpu", "opencv-python-headless"):
    assert pkg in have, (
        f"{pkg} did not stage. Offline extraction (notebook 01) cannot run without it; "
        f"staged prefixes: {sorted(have)}")
total = sum(f.stat().st_size for f in WHEELS.glob("*"))
print(f"{len(list(WHEELS.glob('*.whl')))} wheels, {total/1e9:.1f} GB "
      f"(extraction packages included)")

In [ ]:
# Fail HERE, online, if the staged set cannot satisfy an offline install — not at hour
# three of a GPU session. Checks the things that silently go missing: a torch wheel tagged
# for each interpreter, cudnn (without it torch imports fine and then dies on the first
# conv layer), and an outlines new enough for the API reporter.py actually calls.
#
# Also writes WHEELS_LOCK.txt. The staged wheels ARE the offline environment, and the CPU
# packages are floor-pinned rather than exact-pinned (numpy resolves differently for 3.11
# and 3.12, so one exact list cannot serve both). That means re-running this notebook in
# six weeks stages a different set. The lock file is what makes a training run reproducible
# after the fact, and what the dissertation cites as the environment.
import pathlib, re
WHEELS = pathlib.Path("/kaggle/working/wheels")
names = sorted(p.name for p in WHEELS.glob("*.whl"))
for pyver, cp in (("3.11", "cp311"), ("3.12", "cp312")):
    hits = [n for n in names if n.startswith("torch-") and cp in n]
    assert hits, f"no torch wheel tagged {cp} — offline python {pyver} would have nothing"
    print(f"  py{pyver}: {hits[0]}")
cudnn = [n for n in names if n.startswith("nvidia_cudnn_cu12-")]
assert cudnn, "nvidia-cudnn-cu12 missing — torch would import, then fail on conv layers"
print("  cudnn:", *cudnn)

out = [n for n in names if n.startswith("outlines-")]
assert out, "outlines missing — Agent 4 cannot do grammar-constrained decoding"
major = int(out[0].split("-")[1].split(".")[0])
assert major >= 1, (f"staged {out[0]}: reporter.py calls outlines.from_transformers, "
                    "a 1.x-only API, so this would ImportError offline")
print(f"  outlines: {out[0]} (major {major} >= 1, has from_transformers)")

lock = sorted({f"{m.group(1)}=={m.group(2)}" for n in names
               if (m := re.match(r"(.+?)-([0-9][^-]*)-", n))})
(WHEELS / "WHEELS_LOCK.txt").write_text("\n".join(lock) + "\n")
print(f"  wrote WHEELS_LOCK.txt ({len(lock)} distinct package versions)")

pypi_torch = [n for n in names if n.startswith("torch-") and "+cu128" not in n
              and "%2Bcu128" not in n]
if pypi_torch:
    print("  NOTE: non-cu128 torch also present (wastes space; check the constraint):",
          *pypi_torch)

In [ ]:
# Weights: copy the two OSNet checkpoints out of the repo dataset and PROVE they arrived.
# A Git-LFS pointer is a 130-byte text file that fails at hour three of an offline
# session; this cell is where that mistake gets caught instead.
#
# The loop is deliberately NOT the whole check. On the first real run this cell printed
# NOTHING and the notebook carried on: the uploaded dataset had no weights/ directory, so
# glob() matched nothing, the body never executed, and a missing-asset bug rendered as
# silence. An empty iteration is not a pass. So: name the files expected, diff against
# what actually landed, and report the shortfall.
#
# Not fatal by itself — notebooks 03/04 never load these (ReID tau=0.3546 is already
# fitted and committed in results/reid_eval.md), so this raises only if the directory is
# there but unusable, and otherwise prints a WARNING loud enough to act on.
import shutil, pathlib
_init = sorted(pathlib.Path("/kaggle/input").glob("**/src/behaviorsense/__init__.py"))
CODE = _init[0].parent.parent.parent if _init else pathlib.Path("/kaggle/input/__missing__")
W = pathlib.Path("/kaggle/working/weights"); W.mkdir(parents=True, exist_ok=True)
EXPECTED = {"osnet_ain_x1_0_msmt17.pth", "osnet_ain_x1_0_imagenet.pth"}

src_dir = CODE / "weights"
found = sorted(src_dir.glob("*.pth")) if src_dir.is_dir() else []
for f in found:
    shutil.copy(f, W / f.name)
    mb = (W / f.name).stat().st_size / 1e6
    assert mb > 1, f"{f.name} is {mb:.2f} MB - a pointer file, not weights"
    print(f"  {f.name}: {mb:.1f} MB")

missing = EXPECTED - {f.name for f in found}
if missing:
    print(f"  WARNING: {len(missing)} OSNet checkpoint(s) absent from the dataset: "
          f"{sorted(missing)}")
    print(f"           looked in {src_dir} (exists: {src_dir.is_dir()})")
    print("           Not blocking: notebooks 03/04 do not load these, and the ReID")
    print("           numbers are already measured (results/reid_eval.md). Re-upload")
    print("           behaviorsense-code with weights/ included if you want them staged.")
else:
    print(f"  all {len(EXPECTED)} OSNet checkpoints staged")

In [ ]:
# RTMO-L ONNX: used ONLINE by notebooks 01/02 for pose extraction, and checked by the
# offline preflight. Primary source is the openmmlab CDN; fallback is rtmlib's cache.
import urllib.request, zipfile, pathlib, shutil
W = pathlib.Path("/kaggle/working/weights")
URL = ("https://download.openmmlab.com/mmpose/v1/projects/rtmo/onnx_sdk/"
       "rtmo-l_16xb16-600e_body7-640x640-b37118ce_20231211.zip")
try:
    dst = pathlib.Path("/tmp/rtmo.zip")
    urllib.request.urlretrieve(URL, dst)
    with zipfile.ZipFile(dst) as z:
        onnx = [n for n in z.namelist() if n.endswith(".onnx")]
        z.extract(onnx[0], "/tmp/rtmo")
    shutil.copy(pathlib.Path("/tmp/rtmo") / onnx[0], W / "rtmo-l.onnx")
    print("rtmo-l.onnx:", (W / "rtmo-l.onnx").stat().st_size / 1e6, "MB")
except Exception as exc:
    print("CDN failed:", exc)
    import subprocess
    subprocess.run(["pip", "install", "-q", "rtmlib", "onnxruntime"], check=True)
    from rtmlib import RTMO   # first construction downloads the onnx to ~/.cache
    RTMO(mode="performance", backend="onnxruntime", device="cpu")
    cached = list(pathlib.Path.home().glob(".cache/**/rtmo*[!.zip]"))
    onnx = [p for p in cached if p.suffix == ".onnx"]
    assert onnx, f"no cached onnx found in {cached}"
    shutil.copy(onnx[0], W / "rtmo-l.onnx")
    print("rtmo-l.onnx staged via rtmlib cache")

In [ ]:
# OPTIONAL extras — failures here do not block training (preflight marks both optional):
#   rtdetr-l.onnx        Agent 1 object context (serving-time, not training-time)
#   stgcnpp_ntu60_*.pth  init stretch goal only: our self-contained STGCNpp trains from
#                        scratch, and loading PYSKL checkpoints would need a key-mapping
#                        shim that does not exist yet. Stated honestly rather than staged
#                        as if it were consumed.
import shutil
try:
    import subprocess
    subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
    from ultralytics import RTDETR
    RTDETR("rtdetr-l.pt").export(format="onnx", imgsz=640)
    shutil.copy("rtdetr-l.onnx", "/kaggle/working/weights/rtdetr-l.onnx")
    print("rtdetr-l.onnx staged")
except Exception as exc:
    print(f"RT-DETR skipped (optional): {exc}")

In [ ]:
# Manifest + inventory. Notebook 03's preflight re-verifies this offline.
import pathlib, json
W = pathlib.Path("/kaggle/working/weights")
manifest = {f.name: f.stat().st_size for f in sorted(W.glob("*")) if f.name != "MANIFEST.json"}
(W / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
for name, size in manifest.items():
    print(f"  {name:<40} {size/1e6:>9.1f} MB")
print()
# rtmo-l.onnx is the one asset in this notebook that notebooks 01/02 cannot proceed
# without. An empty or pose-less manifest must not be followed by a cheerful
# "Save Version" — that is how an unusable dataset gets published as if it were fine.
assert manifest, "no weights staged at all - do not Save Version, nothing would be in it"
assert "rtmo-l.onnx" in manifest, (
    f"rtmo-l.onnx missing - notebooks 01/02 cannot extract pose without it. "
    f"Staged: {sorted(manifest)}")
# The staged onnxruntime-gpu decides whether extraction runs on the GPU at all. 1.27+ is
# built against CUDA 13 and silently falls back to CPU on Kaggle's CUDA 12 image, so the
# version is asserted HERE, where it can still be fixed, rather than discovered offline.
ort_whl = sorted(pathlib.Path("/kaggle/working/wheels").glob("onnxruntime_gpu-*.whl"))
assert ort_whl, "onnxruntime-gpu did not stage - notebook 01 cannot use the GPU"
for w in ort_whl:
    ver = w.name.split("-")[1]
    major, minor = (int(x) for x in ver.split(".")[:2])
    assert (major, minor) <= (1, 26), (
        f"staged {w.name}: onnxruntime-gpu >= 1.27 is a CUDA 13 build. Kaggle runs CUDA 12, "
        "so the provider fails to load (libcublasLt.so.13) and RTMO extracts on CPU at "
        "~1/50th speed. Pin onnxruntime-gpu==1.26.0 in the extraction cell.")
    print(f"  {w.name}  (CUDA 12 build, correct)")

print()
print("Save Version -> UPDATE your existing wheels+weights dataset (e.g. behavioursense-WW).")
print("Both folders can live in ONE dataset; the notebooks resolve them by content,")
print("so the dataset name does not matter. Do not create a second copy - it costs quota")
print("and the resolver would then have two candidates to choose between.")